# Sandbox & Python Execution Architecture Demo

This notebook demonstrates **genai-tk's** decoupled sandbox and Python execution subsystems:

1. **Layer 1: Isolation Runtime & Factory (`SandboxBackendFactory`)** — Pluggable runtime backends (`local`, `docker`/`aio_sandbox`, `e2b`, `modal`) resolved via YAML qualified class paths or programmatic registry.
2. **Layer 2: Generic Protocol (`SandboxBackendProtocol`)** — Standard runtime contract for shell execution and file operations (not Python-specific).
3. **Layer 3: Python Action Executors (`BasePythonExecutor`)** — In-process AST evaluation (`LocalPythonExecutor`) vs Sandboxed persistent execution (`SandboxedPythonExecutor` / `DockerPythonExecutor`).
4. **Tool Integration (`PythonExecutorTool`)** — LangChain tool wrapper with CodeAct `final_answer(value)` support.

See also: [docs/sandbox_support.md](../docs/sandbox_support.md), [docs/codeact.md](../docs/codeact.md).

## 1. In-Process AST Python Execution (`LocalPythonExecutor`)

`LocalPythonExecutor` parses and walks Python AST safely without `eval()`. It maintains a persistent namespace across invocations, enforces import allow-lists, and captures standard output.

In [ ]:
from genai_tk.agents.tools.python_executor import (
    BasePythonExecutor,
    LocalPythonExecutor,
    create_python_executor_tool,
)

# 1. Instantiate the in-process AST executor
executor = LocalPythonExecutor(additional_authorized_imports=["math"])
assert isinstance(executor, BasePythonExecutor), "Must conform to BasePythonExecutor protocol"

# Turn 1: Define variables and compute intermediate results
res1 = executor("x = 42\ny = 10\nprint(f'Variables set: x={x}, y={y}')\nx * 2")
print("Turn 1 Output:", res1.output)
print("Turn 1 Logs:\n" + res1.logs)

# Turn 2: Verify state persistence across execution turns
res2 = executor("import math\nresult = math.sqrt(x + y + 2)\nprint('Computed square root:', result)\nresult")
print("Turn 2 Output:", res2.output)
print("Turn 2 Logs:\n" + res2.logs)

## 2. Nested Tool Integration & CodeAct Protocol

Agent tools (such as search or calculation functions) can be nested directly inside the executor's namespace. Calling `final_answer(value)` stops execution and sets `is_final_answer = True`.

In [ ]:
def lookup_capital(country: str) -> str:
    """Sample host function exposed to the Python executor."""
    capitals = {"France": "Paris", "Germany": "Berlin", "Japan": "Tokyo"}
    return capitals.get(country, "Unknown")

codeact_executor = LocalPythonExecutor(tools={"lookup_capital": lookup_capital})

script = """
cap = lookup_capital('France')
print(f'Lookup returned: {cap}')
final_answer(f'The capital of France is {cap}')
"""

res = codeact_executor(script)
print("Final Answer Detected:", res.is_final_answer)
print("Output:", res.output)
print("Logs:\n" + res.logs)

## 3. Pluggable Sandbox Backend Factory (`SandboxBackendFactory`)

`SandboxBackendFactory` allows runtime backends to be registered and resolved by name. Custom backends (e.g. E2B, Modal, Daytona) implement `SandboxBackendProtocol` and register via qualified class paths.

In [ ]:
from genai_tk.agents.sandbox.factory import BUILTIN_SANDBOX_BACKENDS, SandboxBackendFactory

print("Built-in Sandbox Backends:")
for name, cls_path in BUILTIN_SANDBOX_BACKENDS.items():
    print(f" - {name:15s} -> {cls_path}")

# Register a custom backend dynamically or via YAML
class MockCustomCloudBackend:
    """Example custom cloud sandbox (e.g. Modal / E2B) implementing protocol."""
    def __init__(self, **kwargs):
        self.kwargs = kwargs
        self.id = "mock-cloud-sandbox-123"

    async def aexecute(self, command: str, timeout: int = 30):
        return type("ExecRes", (), {"output": f"Executed on cloud: {command}", "exit_code": 0})()

    async def awrite(self, path: str, content: str):
        return type("WriteRes", (), {"bytes_written": len(content)})()

# Register custom backend (ImportResolver uses dot notation: "module.path.ClassName")
SandboxBackendFactory.register("mock_cloud", f"{MockCustomCloudBackend.__module__}.{MockCustomCloudBackend.__name__}")

# Resolve backend class path
resolved_path = SandboxBackendFactory.get_backend_class_path("mock_cloud")
print(f"\nResolved 'mock_cloud' class path: {resolved_path}")
backend_instance = SandboxBackendFactory.create("mock_cloud", api_key="sk-test-123")
print("Instantiated Custom Backend:", backend_instance, "with ID:", backend_instance.id)

## 4. Unified Sandbox Manager (`SandboxManager`)

`SandboxManager` provides singleton lifecycle governance and context variable binding across agent turns to avoid container/session churn.

In [ ]:
from genai_tk.agents.sandbox.manager import DockerSandboxManager, SandboxManager

mgr = SandboxManager.singleton()
print("SandboxManager Singleton:", mgr)
print("Docker Available on Host:", SandboxManager.is_docker_available())

# Backward compatibility alias
assert DockerSandboxManager is SandboxManager

## 5. Sandboxed Python Executor (`SandboxedPythonExecutor`)

`SandboxedPythonExecutor` (aliased to `DockerPythonExecutor` for full backward compatibility) decouples execution from Docker specifics by communicating through standard `SandboxBackendProtocol` methods (`awrite`, `aexecute`) and supporting host tool bridges.

In [ ]:
from genai_tk.agents.tools.python_executor import DockerPythonExecutor, SandboxedPythonExecutor

assert SandboxedPythonExecutor is DockerPythonExecutor, "Backward-compatible alias verified"

# Instantiate sandboxed executor with local fallback enabled
sandboxed_exec = SandboxedPythonExecutor(
    fallback_to_local=True,
    additional_authorized_imports=["math"],
)

# Test synchronous and asynchronous execution
sync_result = sandboxed_exec("a = 15\nb = 30\na + b")
print("Execution Result:", sync_result.output)

## 6. LangChain Tool Wrapper (`PythonExecutorTool`)

The `PythonExecutorTool` exposes the executor to LangGraph / DeepAgents / ReAct loops with structured observation formatting and async execution support.

In [ ]:
tool = create_python_executor_tool(executor_type="local")
print(f"Tool Name: {tool.name}")
print(f"Tool Description: {tool.description[:100]}...")

# Synchronous tool invocation
tool_output = tool.invoke({"code": "total = sum([1, 2, 3, 4, 5])\nprint(f'Total: {total}')\ntotal"})
print("\nTool Invocation Output:\n" + tool_output)

# Async tool invocation
async_tool_output = await tool.ainvoke({"code": "final_answer('All tests passed successfully!')"})
print("\nAsync Tool Final Answer Output:\n" + async_tool_output)